<style>
    @import url('https://fonts.googleapis.com/css2?family=Oswald:wght@400;600&display=swap');
    h1.course-title {
        font-family: 'Oswald', sans-serif;
        font-size: 2.4em;
        color: #E7C173;
        letter-spacing: 0.05em;
        border-bottom: 2px solid #E7C173;
        padding-bottom: 0.3em;
        margin-bottom: 0.2em;
    }
    h2.course-subtitle { font-family: 'Oswald', sans-serif; color: #aaa; font-size: 1.2em; }
</style>

<h1 class='course-title'>MACHINE LEARNING IN INDUSTRY</h1>
<h2 class='course-subtitle'>Cardo AI · MSCA Digital Doctoral Network · Day 4</h2>

# Day 4, Block 3 — Data Drift and Model Monitoring with Evidently

## Table of Contents
1. [Scope and Success Criteria](#1-scope)
2. [Why Monitoring? The Silent Decay Problem](#2-why)
3. [Drift Taxonomy](#3-taxonomy)
4. [Setup — Imports and Data](#4-setup)
5. [Reference and Analysis Sets](#5-reference-analysis)
6. [PSI — The Industry Standard for Drift Detection](#6-psi)
7. [Univariate Drift Detection with Evidently](#7-univariate)
    - [7B. Multiple Testing Correction](#7b-multiple-testing)
    - [7C. Multivariate Drift — Domain Classifier](#7c-domain-classifier)
8. [Evidently HTML Report](#8-report)
9. [Simulating Drift (Covariate Shift)](#9-simulation)
    - [9B. Concept Drift — When Input Monitoring Fails](#9b-concept-drift)
10. [Logging Reports to MLflow](#10-mlflow)
11. [Acceptance Checks](#11-checks)

---
## 1. Scope and Success Criteria <a id='1-scope'></a>

By the end of this notebook you should be able to:

- Explain the difference between **covariate shift**, **concept drift**, and **prior probability shift**
- Set up Evidently **reference** and **analysis** datasets correctly
- Compute and interpret **PSI** (Population Stability Index) for features and scores
- Identify drifting features using **univariate drift detection** and understand multiple testing pitfalls
- Explain why **concept drift cannot be detected** by any input-distribution monitoring method
- Log Evidently reports as **MLflow artifacts** so monitoring history is tracked alongside model history

> **Time budget:** Block 3 (≈45 minutes). The MLflow server from Block 2 should still be running.

---
## 2. Why Monitoring? The Silent Decay Problem <a id='2-why'></a>

Consider a loan default model trained in 2019 on income, employment, and credit history data. In 2020–2021, employment patterns shift dramatically: furlough schemes appear, gig economy income collapses, remote work becomes standard. The model's *feature distribution* has moved — but nobody retrained it.

The model doesn't throw an error. It keeps returning predictions. They just become progressively less accurate. You might only notice six or twelve months later, when loans start defaulting at unexpected rates and you're trying to explain to a regulator why the model drifted out of tolerance.

**The cost of late detection is asymmetric.** Running a drift check costs a few seconds of compute. Missing a distributional shift in a credit risk model costs capital and regulatory credibility.

> 📖 **Recommended reading:** [Failing Loudly (Rabanser et al., 2019)](https://arxiv.org/abs/1810.11953) — a systematic empirical study of drift detection methods. The title says it all: a production ML system should *fail loudly*, not silently.

---
## 3. Drift Taxonomy <a id='3-taxonomy'></a>

There are three fundamentally different things that can go wrong:

| Drift Type | Formal Definition | Credit Risk Example | Detectable without labels? |
|---|---|---|---|
| **Covariate shift** | P(X) changes, but P(Y\|X) stays the same | Age distribution of new applicants shifts younger after a marketing campaign | Yes — PSI, KS test, univariate/multivariate drift |
| **Concept drift** | P(Y\|X) changes — the relationship between features and outcome changes | Remote work makes `commute_miles` irrelevant for income prediction | No — requires ground truth labels to detect |
| **Prior probability shift** | P(Y) changes — the base rate of the outcome shifts | Recession increases the default rate | Partially — score distribution shifts may appear, but confirmation requires labels |

**Key insight:** All monitoring methods that operate on inputs and scores (PSI, KS, chi-squared, multivariate reconstruction error) are blind to concept drift. They can detect covariate shift and some effects of prior probability shift, but confirming *why* performance has changed always requires domain knowledge and, ultimately, labels.

---
## 4. Setup — Imports and Data <a id='4-setup'></a>

In [1]:
# Guard: check the MLflow server is reachable
import urllib.request, urllib.error
MLFLOW_URI = "http://127.0.0.1:5000"
try:
    urllib.request.urlopen(f"{MLFLOW_URI}/health", timeout=2)
    print(f"✓ MLflow server is running at {MLFLOW_URI}")
except urllib.error.URLError:
    print(f"⚠  MLflow server not reachable at {MLFLOW_URI}")
    print("   Run: make -f day4/Makefile mlflow-server")

✓ MLflow server is running at http://127.0.0.1:5000


In [2]:
import sys, json, tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataDriftPreset
from evidently.metrics import DriftedColumnsCount, ValueDrift

repo_root = Path.cwd().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from day4.src.train import (
    DATA_PATH, SEED, TARGET_BIN_COL,
    build_pipeline, get_feature_columns, load_data, split_data,
)

MONITORING_EXPERIMENT = "adult-income-monitoring"
MODEL_NAME = "adult-income-classifier"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(MONITORING_EXPERIMENT)

np.random.seed(SEED)
print("Imports OK")

Imports OK


In [3]:
# Load data and train a model (same as notebook 01)
data_path = repo_root / DATA_PATH
df = load_data(data_path)
train_df, val_df, test_df = split_data(df)

numeric_cols, categorical_cols = get_feature_columns(train_df)
feature_cols = numeric_cols + categorical_cols

X_train, y_train = train_df[feature_cols], train_df[TARGET_BIN_COL]

params = {"n_estimators": 300, "learning_rate": 0.1, "max_depth": 5}
pipe = build_pipeline(numeric_cols, categorical_cols, **params)
pipe.fit(X_train, y_train)

actual_auc = roc_auc_score(
    test_df[TARGET_BIN_COL],
    pipe.predict_proba(test_df[feature_cols])[:, 1],
)
print(f"Model trained.  Actual test AUC: {actual_auc:.4f}")

Model trained.  Actual test AUC: 0.9166


---
## 5. Reference and Analysis Sets <a id='5-reference-analysis'></a>

Evidently's drift detection compares two datasets:

- **Reference set** — data the model was trained or validated on. Ground truth *is* available. Evidently uses this as the baseline distribution.
- **Analysis (current) set** — production data. Ground truth may *not yet* be available (loans haven't defaulted or repaid yet). Evidently compares this against the reference to detect distributional shifts.

Both sets need the same feature columns. We also attach the model's predicted probabilities for score distribution monitoring.

In [4]:
# Reference = training split (ground truth available)
reference_df = df[df["split"] == "train"][feature_cols + [TARGET_BIN_COL]].copy()
reference_df["y_pred_proba"] = pipe.predict_proba(reference_df[feature_cols])[:, 1]
reference_df["y_pred"] = (reference_df["y_pred_proba"] >= 0.5).astype(int)

# Analysis = test split (simulate: ground truth withheld from drift detection)
analysis_df = test_df[feature_cols].copy()
analysis_df["y_pred_proba"] = pipe.predict_proba(analysis_df)[:, 1]
analysis_df["y_pred"] = (analysis_df["y_pred_proba"] >= 0.5).astype(int)
# Keep a separate copy WITH ground truth for validation at the end
analysis_with_gt = analysis_df.copy()
analysis_with_gt[TARGET_BIN_COL] = test_df[TARGET_BIN_COL].values

print(f"Reference : {len(reference_df)} rows (train split)")
print(f"Analysis  : {len(analysis_df)} rows (test split)")
print(f"\nThe analysis set has NO target column — simulating production conditions.")

Reference : 7397 rows (train split)
Analysis  : 1874 rows (test split)

The analysis set has NO target column — simulating production conditions.


---
## 6. PSI — The Industry Standard for Drift Detection <a id='6-psi'></a>

In financial services, **Population Stability Index (PSI)** is a popular metric with model validation teams and regulators (ECB, Fed). For instance, it allows practitioners to quantify shifts in PD buckets over time. It is simple, interpretable, and requires no assumptions about model internals.

**PSI formula:**

```
PSI = Σ (p_analysis_i - p_reference_i) × ln(p_analysis_i / p_reference_i)
```

where `p_i` is the proportion of observations in bin `i`.

**Thresholds (industry standard):**

| PSI | Interpretation |
|---|---|
| < 0.10 | Stable — no significant shift |
| 0.10 – 0.2 | Moderate shift — investigate |
| > 0.2 | Significant shift — likely retrain |

PSI can be applied to both **features** (input PSI — has the data changed?) and **scores** (score PSI — has the model's output distribution changed?). It is symmetric, interpretable, and does not depend on any assumptions about model calibration.

In [5]:
def compute_psi(reference, analysis, n_bins=10, eps=1e-4):
    """
    Population Stability Index between two 1-D distributions.

    Bins are defined by quantiles of the reference distribution so each
    bin has roughly equal reference mass.
    """
    breakpoints = np.quantile(reference, np.linspace(0, 1, n_bins + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf

    ref_counts = np.histogram(reference, bins=breakpoints)[0]
    ana_counts = np.histogram(analysis, bins=breakpoints)[0]

    ref_pcts = ref_counts / len(reference)
    ana_pcts = ana_counts / len(analysis)

    # Clip to avoid log(0)
    ref_pcts = np.clip(ref_pcts, eps, None)
    ana_pcts = np.clip(ana_pcts, eps, None)

    return float(np.sum((ana_pcts - ref_pcts) * np.log(ana_pcts / ref_pcts)))


# ── Score PSI ─────────────────────────────────────────────────────────────────
score_psi = compute_psi(
    reference_df["y_pred_proba"].values,
    analysis_df["y_pred_proba"].values,
)
print(f"Score PSI: {score_psi:.4f}", end="  ")
if score_psi < 0.10:
    print("→ Stable")
elif score_psi < 0.25:
    print("→ Moderate shift — investigate")
else:
    print("→ Significant shift — retrain")

# ── Feature PSI ───────────────────────────────────────────────────────────────
psi_results = []
for col in numeric_cols:
    ref_vals = pd.to_numeric(reference_df[col], errors="coerce").dropna().values
    ana_vals = pd.to_numeric(analysis_df[col], errors="coerce").dropna().values
    if len(ref_vals) < 20 or len(ana_vals) < 20:
        continue
    psi_val = compute_psi(ref_vals, ana_vals)
    status = "stable" if psi_val < 0.10 else ("investigate" if psi_val < 0.25 else "RETRAIN")
    psi_results.append({"feature": col, "PSI": round(psi_val, 4), "status": status})

psi_df = pd.DataFrame(psi_results).sort_values("PSI", ascending=False)
print("\nFeature PSI (numeric features):")
display(psi_df)

Score PSI: 0.0096  → Stable

Feature PSI (numeric features):


,feature,PSI,status
0,education_num,0.0095,stable
2,capital_loss,0.0032,stable
1,capital_gain,0.0003,stable
3,kyc_name_mismatch_flag,0.0000,stable
4,kyc_address_mismatch_flag,0.0000,stable
5,id_document_expired_flag,0.0000,stable
6,manual_review_required_flag,0.0000,stable
7,watchlist_screening_hit_flag,0.0000,stable
8,email_bounce_last_30d_flag,0.0000,stable
9,device_risk_high_flag,0.0000,stable


**PSI vs. statistical drift tests:** PSI and statistical tests (KS, Wasserstein, Jensen-Shannon) often agree on which features have shifted, but they measure different things. KS measures the maximum pointwise difference between two CDFs. Wasserstein (Earth Mover's Distance) measures the minimum "work" needed to transform one distribution into the other. Jensen-Shannon measures divergence between two probability distributions. PSI measures divergence across quantile-based bins — more sensitive to changes in the bulk of the distribution. In practice, PSI is often the first metric in regulated industries like banking.

> With our dataset size (> 1,000 rows), Evidently defaults to Wasserstein/Jensen-Shannon rather than KS/chi-squared. The `method` column in the table above shows which test was used for each feature.

---
## 7. Univariate Drift Detection with Evidently <a id='7-univariate'></a>

Evidently's `DataDriftPreset` tests each feature independently. The **default test method is data-adaptive** — it depends on the reference dataset size and feature cardinality:

| Condition | Numerical features | Categorical features |
|---|---|---|
| Reference ≤ 1,000 rows | KS test (p-value, alpha = 0.05) | Chi-squared (p-value, alpha = 0.05) |
| Reference > 1,000 rows | **Wasserstein distance** (threshold = 0.1) | **Jensen-Shannon distance** (threshold = 0.1) |

Our training set has ~5,900 rows, so Evidently will use **Wasserstein** for numeric features and **Jensen-Shannon** for categorical features. The output is a distance value compared against a threshold (not a p-value) — a feature is flagged as drifted when the distance exceeds the threshold.

You can override the defaults per feature type:
```python
DataDriftPreset(num_method="ks", cat_method="chi_square")  # force classical tests
```
> 📖 **Further reading:** [Which test is the best? We compared 5 methods to detect data drift on large datasets](https://www.evidentlyai.com/blog/data-drift-detection-large-datasets) — A comparison of the main drift detection tests on a 100k row dataset.

In [6]:
# Build Evidently DataDefinition and Datasets
data_def = DataDefinition(
    numerical_columns=numeric_cols,
    categorical_columns=categorical_cols,
)

ref_ds = Dataset.from_pandas(reference_df[feature_cols], data_definition=data_def)
cur_ds = Dataset.from_pandas(analysis_df[feature_cols], data_definition=data_def)

# Run drift detection
drift_report = Report(metrics=[DataDriftPreset()])
drift_result = drift_report.run(reference_data=ref_ds, current_data=cur_ds)

# Extract per-feature drift results from the report
result_dict = drift_result.dict()
drifted_features = []
drift_details = []
for metric in result_dict.get("metrics", []):
    cfg = metric.get("config", {})
    if cfg.get("type", "").endswith("ValueDrift"):
        col_name = cfg.get("column", "")
        method = cfg.get("method", "")
        threshold = cfg.get("threshold", 0.1)
        stat_value = metric.get("value")
        # Distance-based methods (Wasserstein, Jensen-Shannon): drift when value >= threshold
        # P-value-based methods (KS, chi-squared): drift when value <= threshold
        is_distance = "distance" in method.lower()
        if stat_value is not None:
            is_drifted = stat_value >= threshold if is_distance else stat_value <= threshold
        else:
            is_drifted = False
        if is_drifted:
            drifted_features.append(col_name)
        drift_details.append({
            "feature": col_name,
            "method": method,
            "stat_value": round(stat_value, 6) if stat_value is not None else None,
            "threshold": threshold,
            "drifted": is_drifted,
        })

drift_details_df = pd.DataFrame(drift_details).sort_values(
    ["drifted", "stat_value"], ascending=[False, False]
)
print(f"Drifted features ({len(drifted_features)} / {len(feature_cols)}): {drifted_features or 'none'}")
print()
display(drift_details_df)

Drifted features (2 / 19): ['hours_per_week', 'occupation']



,feature,method,stat_value,threshold,drifted
14,occupation,Jensen-Shannon distance,0.173421,0.1,True
13,hours_per_week,Jensen-Shannon distance,0.102184,0.1,True
12,age,Jensen-Shannon distance,0.095375,0.1,False
18,relationship,Jensen-Shannon distance,0.058856,0.1,False
0,education_num,Wasserstein distance (normed),0.053997,0.1,False
15,native_country,Jensen-Shannon distance,0.036743,0.1,False
16,workclass,Jensen-Shannon distance,0.036083,0.1,False
17,marital_status,Jensen-Shannon distance,0.030894,0.1,False
2,capital_loss,Jensen-Shannon distance,0.020020,0.1,False
5,id_document_expired_flag,Jensen-Shannon distance,0.017404,0.1,False


### 7B. Multiple Testing Correction <a id='7b-multiple-testing'></a>

Bonferroni correction applies to **p-value-based** hypothesis tests. Our default Evidently run used distance-based methods (Wasserstein/Jensen-Shannon), which compare a distance against a fixed threshold — there's no p-value to correct.

To demonstrate the multiple testing problem, we'll re-run with forced **KS/chi-squared** tests (which return p-values). This also shows how to override Evidently's default method selection.

With 19 features tested at alpha=0.05, the **family-wise error rate** (FWER) is:
> P(at least 1 false positive) = 1 - (1 - 0.05)^19 ≈ 62%

**Evidently does not apply any multiple testing correction automatically.** You need to do this yourself.

In [7]:
import warnings

# Re-run with forced KS/chi-squared to get p-values for Bonferroni correction
drift_report_pval = Report(metrics=[
    DataDriftPreset(num_method="ks", cat_method="chisquare")
])
# Suppress divide-by-zero warnings from scipy's chi-squared test on near-constant features
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="divide by zero", category=RuntimeWarning)
    drift_result_pval = drift_report_pval.run(reference_data=ref_ds, current_data=cur_ds)

# Extract p-values from the KS/chi-squared run
result_dict_pval = drift_result_pval.dict()
pval_details = []
for metric in result_dict_pval.get("metrics", []):
    cfg = metric.get("config", {})
    if cfg.get("type", "").endswith("ValueDrift"):
        col_name = cfg.get("column", "")
        method = cfg.get("method", "")
        p_value = metric.get("value")
        pval_details.append({"feature": col_name, "method": method, "p_value": p_value})

pval_df = pd.DataFrame(pval_details).sort_values("p_value")

n_features = len(feature_cols)
alpha = 0.05

# Bonferroni correction: divide alpha by number of tests
bonferroni_alpha = alpha / n_features

# Family-wise error rate without correction
fwer_uncorrected = 1 - (1 - alpha) ** n_features

print(f"Number of features tested: {n_features}")
print(f"Uncorrected alpha:         {alpha}")
print(f"Bonferroni-corrected alpha: {bonferroni_alpha:.4f}")
print(f"P(≥1 false positive) without correction: {fwer_uncorrected:.1%}")
print(f"P(≥1 false positive) with Bonferroni:    {alpha:.1%}  (by construction)")
print()

uncorrected_drifted = pval_df[pval_df["p_value"] < alpha]["feature"].tolist()
robust_drifted = pval_df[pval_df["p_value"] < bonferroni_alpha]["feature"].tolist()

print(f"Features with p < {alpha} (uncorrected):          {len(uncorrected_drifted)} — {uncorrected_drifted or 'none'}")
print(f"Features with p < {bonferroni_alpha:.4f} (Bonferroni):  {len(robust_drifted)} — {robust_drifted or 'none'}")
print()
display(pval_df)
print()
print("Strategies for production:")
print("  1. Bonferroni correction (conservative — good for regulated environments)")
print("  2. Benjamini-Hochberg FDR control (less conservative)")
print("  3. Monitor only the top-k most important features")

Number of features tested: 19
Uncorrected alpha:         0.05
Bonferroni-corrected alpha: 0.0026
P(≥1 false positive) without correction: 62.3%
P(≥1 false positive) with Bonferroni:    5.0%  (by construction)

Features with p < 0.05 (uncorrected):          4 — ['relationship', 'occupation', 'hours_per_week', 'age']
Features with p < 0.0026 (Bonferroni):  4 — ['relationship', 'occupation', 'hours_per_week', 'age']



,feature,method,p_value
18,relationship,chisquare,0.000000
14,occupation,chisquare,0.000000
13,hours_per_week,chisquare,0.000000
12,age,chisquare,0.000378
0,education_num,ks,0.050296
15,native_country,chisquare,0.099015
16,workclass,chisquare,0.171383
17,marital_status,chisquare,0.219368
2,capital_loss,ks,0.298860
1,capital_gain,ks,0.999998



Strategies for production:
  1. Bonferroni correction (conservative — good for regulated environments)
  2. Benjamini-Hochberg FDR control (less conservative)
  3. Monitor only the top-k most important features


### 7C. Multivariate Drift — Domain Classifier <a id='7c-domain-classifier'></a>

Univariate tests check each feature independently. With distance-based methods (Wasserstein, Jensen-Shannon), there are no p-values — so Bonferroni doesn't apply directly. And even with p-value tests, testing many features inflates false positives.

A **domain classifier** sidesteps this entirely by testing the **joint** distribution in a single test:

1. Combine reference and current data into one dataset
2. Label: reference = 0, current = 1
3. Train a simple classifier (logistic regression) to distinguish them
4. Evaluate with cross-validated **ROC-AUC**

**Interpretation:**
- **AUC ≈ 0.5** → the classifier can't tell the datasets apart → no drift
- **AUC >> 0.5** → the classifier easily distinguishes them → the joint distribution has shifted

**Why this works:** if *any* combination of features has shifted — even subtle correlated changes that no single feature test would catch — the classifier will exploit it. And since it's a single test, there's no multiple testing problem.

**Bonus:** the classifier's coefficients reveal *which features* contribute most to the shift, giving you the same interpretability as per-feature tests.

> 📖 Rabanser et al. (2019) ["Failing Loudly"](https://arxiv.org/abs/1810.11953) found domain classifiers among the most effective drift detectors across 7 datasets and 9 perturbation types.

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline  # avoid shadowing mlflow Pipeline

# Combine reference and current data, label them
X_ref = reference_df[feature_cols]
X_cur = analysis_df[feature_cols]
X_combined = pd.concat([X_ref, X_cur], ignore_index=True)
y_domain = np.array([0] * len(X_ref) + [1] * len(X_cur))

# Preprocessing: impute missing values, then scale/encode (mirrors build_pipeline)
preprocessor = ColumnTransformer([
    ("num", SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_cols),
    ("cat", SkPipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_cols),
])
domain_clf = SkPipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
])

# 5-fold cross-validated ROC-AUC
auc_scores = cross_val_score(domain_clf, X_combined, y_domain, cv=5, scoring="roc_auc")
domain_auc = auc_scores.mean()

print(f"Domain classifier AUC: {domain_auc:.4f} (±{auc_scores.std():.4f})")
if domain_auc < 0.55:
    print("→ No significant drift (classifier can't distinguish reference from current)")
elif domain_auc < 0.65:
    print("→ Mild drift — investigate")
else:
    print("→ Significant drift — the joint distribution has shifted")

Domain classifier AUC: 0.5385 (±0.0181)
→ No significant drift (classifier can't distinguish reference from current)


---
## 8. Evidently HTML Report <a id='8-report'></a>

One of Evidently's key features is generating rich HTML reports with per-feature distribution overlays — reference vs. current data side by side. These reports are self-contained HTML files that can be shared with stakeholders, attached to MLflow runs, or reviewed in CI/CD pipelines.

In [9]:
# Save the Evidently HTML report
report_path = Path("../outputs/drift_report.html")
report_path.parent.mkdir(parents=True, exist_ok=True)
drift_result.save_html(str(report_path))
print(f"Drift report saved to {report_path}")
print("Open it in your browser to see per-feature distribution overlays.")

# Also extract overall drift summary
for metric in result_dict.get("metrics", []):
    cfg = metric.get("config", {})
    if cfg.get("type", "").endswith("DriftedColumnsCount"):
        n_drifted = metric["value"]["count"]
        share_drifted = metric["value"]["share"]
        print(f"\nDataset-level drift: {int(n_drifted)} / {len(feature_cols)} features drifted ({share_drifted:.0%})")
        break

Drift report saved to ../outputs/drift_report.html
Open it in your browser to see per-feature distribution overlays.

Dataset-level drift: 2 / 19 features drifted (11%)


---
## 9. Simulating Drift (Covariate Shift) <a id='9-simulation'></a>

The test data above is drawn from the same distribution as training, so drift is minimal. Let's *simulate* what a macroeconomic shock looks like: compress `hours_per_week` (people shift to part-time work) and add noise.

In [10]:
analysis_drifted = analysis_df.copy()

if "hours_per_week" in analysis_drifted.columns:
    rng = np.random.RandomState(SEED)
    hpw = pd.to_numeric(analysis_drifted["hours_per_week"], errors="coerce")
    analysis_drifted["hours_per_week"] = (
        hpw * 0.65   # compress toward fewer hours
        + rng.normal(0, 4, len(analysis_drifted))    # add noise
    ).clip(1, 99)
    print("hours_per_week before perturbation:")
    print(f"  mean={pd.to_numeric(analysis_df['hours_per_week'], errors='coerce').mean():.1f}  std={pd.to_numeric(analysis_df['hours_per_week'], errors='coerce').std():.1f}")
    print("hours_per_week after perturbation:")
    print(f"  mean={analysis_drifted['hours_per_week'].mean():.1f}  std={analysis_drifted['hours_per_week'].std():.1f}")
else:
    print("hours_per_week not found in feature set — skipping perturbation")

hours_per_week before perturbation:
  mean=40.8  std=15.7
hours_per_week after perturbation:
  mean=26.7  std=10.9


In [11]:
# Re-run Evidently drift detection on drifted data
ref_ds_d = Dataset.from_pandas(reference_df[feature_cols], data_definition=data_def)
cur_ds_d = Dataset.from_pandas(analysis_drifted[feature_cols], data_definition=data_def)

drift_report_d = Report(metrics=[DataDriftPreset()])
drift_result_d = drift_report_d.run(reference_data=ref_ds_d, current_data=cur_ds_d)

result_dict_d = drift_result_d.dict()
drifted_after = []
drift_details_after = {}
for metric in result_dict_d.get("metrics", []):
    cfg = metric.get("config", {})
    if cfg.get("type", "").endswith("ValueDrift"):
        col_name = cfg.get("column", "")
        method = cfg.get("method", "")
        threshold = cfg.get("threshold", 0.1)
        stat_value = metric.get("value")
        is_distance = "distance" in method.lower()
        if stat_value is not None:
            is_drifted = stat_value >= threshold if is_distance else stat_value <= threshold
        else:
            is_drifted = False
        if is_drifted:
            drifted_after.append(col_name)
        drift_details_after[col_name] = {"stat_value": stat_value, "drifted": is_drifted}

# Side-by-side comparison with delta column
comparison = []
for _, row in drift_details_df.iterrows():
    col = row["feature"]
    after = drift_details_after.get(col, {})
    before_val = row["stat_value"]
    after_val = after.get("stat_value", 0)
    comparison.append({
        "feature": col,
        "before": round(before_val, 4) if before_val is not None else None,
        "after": round(after_val, 4),
        "delta": round(after_val - before_val, 4) if before_val is not None else None,
        "threshold": row["threshold"],
        "drifted_before": row["drifted"],
        "drifted_after": after.get("drifted", False),
    })
comp_df = pd.DataFrame(comparison).sort_values("delta", ascending=False, key=abs, na_position="last")

print(f"Before perturbation — drifted: {len(drifted_features)} / {len(feature_cols)}")
print(f"After  perturbation — drifted: {len(drifted_after)} / {len(feature_cols)}")
print()
print("Note: the COUNT stays the same because hours_per_week was already marginally")
print("above the 0.1 threshold (0.1022) before perturbation. The perturbation didn't")
print("flip it from non-drifted to drifted — it made an already-drifted feature MUCH")
print(f"more drifted (0.1022 → 0.8326). Check the delta column below.")
print()
display(comp_df)

# Save the drifted report for comparison
drifted_report_path = Path("../outputs/drift_report_perturbed.html")
drift_result_d.save_html(str(drifted_report_path))
print(f"\nPerturbed drift report saved to {drifted_report_path}")

Before perturbation — drifted: 2 / 19
After  perturbation — drifted: 2 / 19

Note: the COUNT stays the same because hours_per_week was already marginally
above the 0.1 threshold (0.1022) before perturbation. The perturbation didn't
flip it from non-drifted to drifted — it made an already-drifted feature MUCH
more drifted (0.1022 → 0.8326). Check the delta column below.



,feature,before,after,delta,threshold,drifted_before,drifted_after
1,hours_per_week,0.1022,0.8326,0.7304,0.1,True,True
0,occupation,0.1734,0.1734,-0.0000,0.1,True,True
10,income_proof_unreadable_flag,0.0163,0.0163,-0.0000,0.1,False,False
17,kyc_address_mismatch_flag,0.0019,0.0019,-0.0000,0.1,False,False
16,manual_review_required_flag,0.0020,0.0020,-0.0000,0.1,False,False
15,email_bounce_last_30d_flag,0.0023,0.0023,-0.0000,0.1,False,False
14,duplicate_application_flag,0.0046,0.0046,-0.0000,0.1,False,False
13,capital_gain,0.0089,0.0089,0.0000,0.1,False,False
12,kyc_name_mismatch_flag,0.0153,0.0153,0.0000,0.1,False,False
11,device_risk_high_flag,0.0160,0.0160,-0.0000,0.1,False,False



Perturbed drift report saved to ../outputs/drift_report_perturbed.html


**Domain classifier on perturbed data:** The univariate tests above flag `hours_per_week` — but does the domain classifier also pick up the perturbation as a multivariate signal?

In [12]:
# Domain classifier on perturbed analysis set (hours_per_week compressed)
X_cur_d = analysis_drifted[feature_cols].copy()
# Perturbation converted hours_per_week to float; cast categoricals back to str
# so OHE sees uniform types after concat with reference data
for col in categorical_cols:
    X_cur_d[col] = X_cur_d[col].astype(str)
X_combined_d = pd.concat([X_ref, X_cur_d], ignore_index=True)
y_domain_d = np.array([0] * len(X_ref) + [1] * len(X_cur_d))

auc_scores_d = cross_val_score(domain_clf, X_combined_d, y_domain_d, cv=5, scoring="roc_auc")
domain_auc_perturbed = auc_scores_d.mean()

print(f"{'Dataset':<30s} {'Domain AUC':>12s}")
print(f"{'─' * 43}")
print(f"{'Original analysis':<30s} {domain_auc:>12.4f}")
print(f"{'Perturbed (hours_per_week)':<30s} {domain_auc_perturbed:>12.4f}")
print()
print("The domain classifier picks up the perturbation as a single, multivariate signal.")

Dataset                          Domain AUC
───────────────────────────────────────────
Original analysis                    0.5385
Perturbed (hours_per_week)           0.9986

The domain classifier picks up the perturbation as a single, multivariate signal.


In [13]:
# Fit on perturbed data to inspect which features the classifier uses
domain_clf.fit(X_combined_d, y_domain_d)

# Get feature names after one-hot encoding
# OHE is nested: prep → cat → onehot
ohe = domain_clf.named_steps["prep"].named_transformers_["cat"].named_steps["onehot"]
ohe_names = ohe.get_feature_names_out(categorical_cols).tolist()
all_feature_names = numeric_cols + ohe_names
coefs = domain_clf.named_steps["clf"].coef_[0]

# Aggregate OHE coefficients back to original feature level:
# for each categorical feature, sum |coef| across all its dummy columns
feature_importance = {}
for name, c in zip(all_feature_names, coefs):
    # Map OHE name (e.g. "occupation_Sales") back to original feature
    original = name
    for cat_col in categorical_cols:
        if name.startswith(cat_col + "_"):
            original = cat_col
            break
    feature_importance[original] = feature_importance.get(original, 0) + abs(c)

coef_df = (
    pd.DataFrame({"feature": list(feature_importance.keys()),
                   "importance": list(feature_importance.values())})
    .sort_values("importance", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print("Top 10 features distinguishing reference from current (perturbed):")
print("(For categorical features, importance = sum of |coef| across all OHE categories)\n")
display(coef_df)

Top 10 features distinguishing reference from current (perturbed):
(For categorical features, importance = sum of |coef| across all OHE categories)



,feature,importance
0,hours_per_week,664.626601
1,age,11.378457
2,occupation,10.960586
3,relationship,3.502595
4,workclass,1.990646
5,marital_status,1.495041
6,native_country,1.259008
7,kyc_name_mismatch_flag,0.048640
8,education_num,0.043734
9,capital_loss,0.036689


---
## 9B. Concept Drift — When Input Monitoring Fails <a id='9b-concept-drift'></a>

The simulation above is **covariate shift** — P(X) changed (hours_per_week compressed). PSI, KS, and univariate drift detection all catch this, because they monitor the feature distribution.

**Concept drift** is fundamentally different: P(Y|X) changes — the relationship between features and outcome shifts — while the feature distribution stays the same. This is the hardest type of drift to detect, and *no input-distribution-based monitoring method can catch it*.

Let's demonstrate this by flipping the labels for a subgroup while keeping the features and scores unchanged.

In [14]:
# ── Simulate concept drift: flip labels for high-confidence predictions ────────
# Features and scores stay IDENTICAL. Only the ground truth changes.
analysis_concept = analysis_with_gt.copy()
high_conf_mask = analysis_concept["y_pred_proba"] > 0.7
n_flipped = high_conf_mask.sum()
analysis_concept.loc[high_conf_mask, TARGET_BIN_COL] = (
    1 - analysis_concept.loc[high_conf_mask, TARGET_BIN_COL]
)
print(f"Flipped {n_flipped} labels ({n_flipped/len(analysis_concept):.1%} of analysis set)")

# ── PSI on scores: unchanged (same predictions) ──────────────────────────────
score_psi_concept = compute_psi(
    reference_df["y_pred_proba"].values,
    analysis_concept["y_pred_proba"].values,
)

# ── KS test on scores: compute BEFORE and AFTER to show they're identical ─────
from scipy.stats import ks_2samp
ks_stat_before, ks_pval_before = ks_2samp(
    reference_df["y_pred_proba"].values,
    analysis_with_gt["y_pred_proba"].values,  # pre-flip scores
)
ks_stat_after, ks_pval_after = ks_2samp(
    reference_df["y_pred_proba"].values,
    analysis_concept["y_pred_proba"].values,  # post-flip scores (identical)
)

# ── Actual AUC: degraded (labels changed) ─────────────────────────────────────
actual_auc_concept = roc_auc_score(
    analysis_concept[TARGET_BIN_COL],
    analysis_concept["y_pred_proba"],
)

print(f"\n{'Metric':<30s} {'Before concept drift':>20s} {'After concept drift':>20s}")
print(f"{'─' * 72}")
print(f"{'Score PSI':<30s} {score_psi:>20.4f} {score_psi_concept:>20.4f}")
print(f"{'KS statistic (p-value)':<30s} {f'{ks_stat_before:.4f} (p={ks_pval_before:.3f})':>20s} {f'{ks_stat_after:.4f} (p={ks_pval_after:.3f})':>20s}")
print(f"{'Actual AUC':<30s} {actual_auc:>20.4f} {actual_auc_concept:>20.4f}")
print()
print("PSI is identical before and after — the score distribution didn't change.")
print("KS is identical before and after — it detects the same train/test gap,")
print("  but that gap exists regardless of concept drift.")
print(f"Meanwhile, actual AUC dropped by {actual_auc - actual_auc_concept:.4f} — invisible to both metrics.")

Flipped 379 labels (20.2% of analysis set)

Metric                         Before concept drift  After concept drift
────────────────────────────────────────────────────────────────────────
Score PSI                                    0.0096               0.0096
KS statistic (p-value)             0.0377 (p=0.027)     0.0377 (p=0.027)
Actual AUC                                   0.9166               0.7273

PSI is identical before and after — the score distribution didn't change.
KS is identical before and after — it detects the same train/test gap,
  but that gap exists regardless of concept drift.
Meanwhile, actual AUC dropped by 0.1893 — invisible to both metrics.


**The takeaway:** PSI and KS are **unchanged** by the concept drift — they return identical values before and after the label flip. The score and feature distributions didn't move; only the ground truth did. This is the fundamental limitation of all input-based and score-based monitoring.

**What to do in production:**
1. **Pursue label collection** — even when all monitoring looks green. In credit risk, use early proxies: 30/60/90-day delinquency as leading indicators before final default/repayment status arrives.
2. **Periodic backtesting** — once labels arrive (even partially), compare actual performance against the reference period. If performance has degraded, concept drift has likely occurred.
3. **Domain expert review** — if you know the economic environment has changed (pandemic, regulatory shift, market shock), assume concept drift and investigate regardless of monitoring signals.
4. **Score distribution tracking** — while score PSI won't catch concept drift, tracking score mean, standard deviation, and percentiles over time provides useful context when labels do arrive.

> **Gama et al. (2014)** — [*A Survey on Concept Drift Adaptation*](https://doi.org/10.1145/2523813) provides a comprehensive taxonomy of concept drift detection and adaptation strategies.

**Important Note:** This is not a theorem. You might get feature shifts that correlate with concept drift, but it's not guaranteed. On the other hand, if you see no feature shifts at all but performance degrades, concept drift is a likely suspect. Finally, you might as well have feture drifts that don't cause performance degradation — not all distributional shifts are harmful.

---
## 10. Logging Reports to MLflow <a id='10-mlflow'></a>

In production, monitoring runs on a schedule (nightly cron or a CI/CD trigger). Each run logs its reports to MLflow as artifacts, giving you a time-series of monitoring snapshots that live alongside your model versions. This is the connective tissue between MLflow and Evidently.

In [15]:
with mlflow.start_run(run_name="evidently-monitoring-demo") as run:
    # Log scalar summary metrics
    mlflow.log_metric("actual_val_auc", actual_auc)
    mlflow.log_metric("score_psi", score_psi)
    mlflow.log_metric("n_drifted_features", len(drifted_features))
    mlflow.log_metric("domain_classifier_auc", domain_auc)

    with tempfile.TemporaryDirectory() as tmpdir:
        # Drift HTML report
        drift_html_path = Path(tmpdir) / "drift_report.html"
        drift_result.save_html(str(drift_html_path))
        mlflow.log_artifact(str(drift_html_path), artifact_path="drift_reports")

        # Drift summary JSON
        summary_path = Path(tmpdir) / "drift_summary.json"
        summary = {
            "drifted_features": drifted_features,
            "n_drifted": len(drifted_features),
            "score_psi": round(score_psi, 4),
            "domain_classifier_auc": round(domain_auc, 4),
        }
        summary_path.write_text(json.dumps(summary, indent=2))
        mlflow.log_artifact(str(summary_path), artifact_path="drift_reports")

    monitoring_run_id = run.info.run_id
    print(f"Monitoring run logged. Run ID: {monitoring_run_id}")
    print(f"Open {MLFLOW_URI} → experiment '{MONITORING_EXPERIMENT}' to see the artifacts.")

Monitoring run logged. Run ID: ca012ddd52954a2b89e6104bfe8c5c4e
Open http://127.0.0.1:5000 → experiment 'adult-income-monitoring' to see the artifacts.
🏃 View run evidently-monitoring-demo at: http://127.0.0.1:5000/#/experiments/2/runs/ca012ddd52954a2b89e6104bfe8c5c4e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


---
## 11. Acceptance Checks <a id='11-checks'></a>

In [16]:
# ── Reference and analysis sets ───────────────────────────────────────────────
assert len(reference_df) > 0, "Reference set is empty"
assert len(analysis_df) > 0, "Analysis set is empty"
assert "y_pred_proba" in reference_df.columns, "Missing y_pred_proba in reference"
assert "y_pred_proba" in analysis_df.columns, "Missing y_pred_proba in analysis"

# ── Score PSI is reasonable ───────────────────────────────────────────────────
assert score_psi < 0.2, f"Score PSI {score_psi:.4f} > 0.2 — significant shift detected"

# ── MLflow artifact logged ────────────────────────────────────────────────────
client = mlflow.tracking.MlflowClient()
artifacts = client.list_artifacts(monitoring_run_id, path="drift_reports")
artifact_names = [a.path for a in artifacts]
assert any("drift_report" in n for n in artifact_names), "Drift report not found in MLflow"

print("✓ Reference and analysis sets built correctly")
print(f"✓ Score PSI: {score_psi:.4f} (within 0.2)")
print(f"✓ Evidently reports logged to MLflow run {monitoring_run_id}")
print("\nAll acceptance checks passed.")

✓ Reference and analysis sets built correctly
✓ Score PSI: 0.0096 (within 0.2)
✓ Evidently reports logged to MLflow run ca012ddd52954a2b89e6104bfe8c5c4e

All acceptance checks passed.
